In [ ]:
# Setup
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from data import load_data, RAW_DIR
from utils import (save_fig, plot_total_sales_over_time,
                   plot_seasonality, plot_promo_effect)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
sns.set_style('whitegrid')

print(f"Raw data dir: {RAW_DIR}")

In [ ]:
# 1. Load data
df = load_data()
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"Stores: {df['Store'].nunique()}")
df.head()

In [ ]:
# 2. Basic stats and missing values
print(df.dtypes)
print()
print("Missing values per column:")
print(df.isna().sum())

In [ ]:
# 3. Total daily sales over time
plot_total_sales_over_time(df)
save_fig('01_total_sales_over_time.png')
plt.show()

In [ ]:
# 4. Day-of-week and month seasonality
axes = plot_seasonality(df)
save_fig('02_seasonality.png')
plt.show()

In [ ]:
# 5. Promo effect on sales (open stores only)
plot_promo_effect(df)
save_fig('03_promo_effect.png')
plt.show()

promo_lift = (df[df.Open==1].groupby('Promo')['Sales'].mean()[1]
              / df[df.Open==1].groupby('Promo')['Sales'].mean()[0] - 1)
print(f"Promo lift: {promo_lift:.1%}")

In [ ]:
# 6. Store-level variation
store_avg = df[df.Open==1].groupby('Store')['Sales'].mean().sort_values()
fig, ax = plt.subplots(figsize=(11, 4))
store_avg.plot(kind='bar', ax=ax, color='#216b91')
ax.set_title("Average daily sales per store (open days only)")
ax.set_ylabel("Average sales")
ax.set_xlabel("Store")
save_fig('04_store_level_variation.png')
plt.show()
print(f"Lowest-avg store: {store_avg.index[0]} ({store_avg.iloc[0]:.0f})")
print(f"Highest-avg store: {store_avg.index[-1]} ({store_avg.iloc[-1]:.0f})")

In [ ]:
# 7. Save a cleaned copy for the modelling notebook
from data import save_processed
out = save_processed(df, name='rossmann_clean.csv')
print(f"Saved cleaned data → {out}")